# Homework 3: Machine Learning for Classification

Machine Learning Zoomcamp 2026 — Module 3

Dataset: `course_lead_scoring_2026.csv`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score, accuracy_score

df = pd.read_csv("../datasets/course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

Check missing values, fill categorical with `'NA'`, numerical with `0.0`.

In [ ]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [ ]:
for c in categorical:
    df[c] = df[c].fillna('NA')
for c in numerical:
    df[c] = df[c].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Q1. Mode of `industry`

What's the most frequent value?

In [ ]:
df['industry'].mode()[0]

'technology'

## Q2. Correlation matrix

Which pair of numerical features has the biggest correlation?

In [ ]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


## Split the data

Exact calls given in the homework.

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Q3. Mutual information

Between `converted` and each categorical variable, training set only. Which has the biggest score?

In [ ]:
def mi_score(series):
    return round(mutual_info_score(series, y_train), 2)

df_train[categorical].apply(mi_score).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Q4. Logistic regression

One-hot encode with `DictVectorizer`, fit with the given parameters, report validation accuracy.

In [ ]:
features = categorical + numerical

def vectorize(df_a, df_b, feats):
    dv = DictVectorizer(sparse=False)
    X_a = dv.fit_transform(df_a[feats].to_dict(orient='records'))
    X_b = dv.transform(df_b[feats].to_dict(orient='records'))
    return X_a, X_b

X_train, X_val = vectorize(df_train, df_val, features)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
acc_full = accuracy_score(y_val, y_pred)
round(acc_full, 2)

0.65

## Q5. Feature elimination

Drop each candidate feature, retrain, compare validation accuracy to the Q4 baseline (unrounded).

In [ ]:
candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
diffs = {}

for feat in candidates:
    feats_wo = [f for f in features if f != feat]
    X_train_wo, X_val_wo = vectorize(df_train, df_val, feats_wo)
    model_wo = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_wo.fit(X_train_wo, y_train)
    acc_wo = accuracy_score(y_val, model_wo.predict(X_val_wo))
    diffs[feat] = acc_full - acc_wo

diffs

{'lead_source': 0.0030000000000000027,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004}

In [ ]:
min(diffs, key=lambda f: abs(diffs[f]))

'number_of_courses_viewed'

## Q6. Regularized logistic regression

Sweep `C`, accuracy rounded to 3 decimals, pick the best (smallest `C` on ties).

In [ ]:
results = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model_c = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)
    results[C] = round(accuracy_score(y_val, model_c.predict(X_val)), 3)

results

{1e-06: 0.598, 1e-05: 0.598, 0.0001: 0.613, 0.001: 0.645}

In [ ]:
max(results, key=lambda c: (results[c], -c))

0.001